# Exercise 4 — Customer Support Reply Generator

## Objective
Generate a consistent, professional customer support reply.

## Inputs
- customer_message
- company_name
- max_words

## Expected Output
A polite and helpful reply that stays within the specified word limit.

In [13]:
%pip install groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [14]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

client = Groq(api_key=api_key)
MODEL = "openai/gpt-oss-120b"

print("Groq client initialized successfully.")

Groq client initialized successfully.


In [15]:
customer_message = """
I ordered a laptop charger five days ago, but it has not arrived.
My order number is ORD12345. Please help me.
"""

company_name = "ABC Store"
max_words = 60

print("Company:", company_name)
print("Customer message:", customer_message)
print("Maximum words:", max_words)

Company: ABC Store
Customer message: 
I ordered a laptop charger five days ago, but it has not arrived.
My order number is ORD12345. Please help me.

Maximum words: 60


In [16]:
prompt_template = """
You are a customer support representative for {company_name}.

Write a helpful, professional, polite, and empathetic reply
to the customer's message.

Requirements:
1. Use a consistent professional and friendly tone.
2. Keep the reply within {max_words} words.
3. Acknowledge the customer's concern.
4. Do not invent order status, tracking details, refunds, or promises.
5. If more information is needed, explain the next appropriate step.
6. Return only the reply, without a heading or word count.

Customer message:
{customer_message}
"""

prompt = prompt_template.format(
    company_name=company_name,
    max_words=max_words,
    customer_message=customer_message
)

print("Prompt created successfully.")

Prompt created successfully.


In [19]:
def generate_support_reply(customer_message, company_name, max_words):
    if not customer_message.strip():
        raise ValueError("Customer message cannot be empty.")

    if not company_name.strip():
        raise ValueError("Company name cannot be empty.")

    if max_words <= 0:
        raise ValueError("Maximum words must be greater than zero.")

    prompt = f"""
You are a customer support representative for {company_name}.

Write a professional, polite, empathetic reply to this customer.

Rules:
- Use a friendly and professional tone.
- Acknowledge the customer's concern.
- Do not invent order status, tracking details, refunds, or promises.
- Stay within {max_words} words.
- Return only the reply, without a heading.

Customer message:
{customer_message}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Write concise, helpful customer support replies. "
                    "Follow all instructions and return the reply text."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_completion_tokens=1000
    )

    reply = response.choices[0].message.content

    if not reply or not reply.strip():
        raise ValueError(
            "The model returned an empty reply. "
            f"Finish reason: {response.choices[0].finish_reason}. "
            "Check the response before retrying."
        )

    return reply.strip()

In [21]:
reply = generate_support_reply(
    customer_message,
    company_name,
    max_words
)

print("CUSTOMER SUPPORT REPLY")
print("-" * 30)
print(reply)

CUSTOMER SUPPORT REPLY
------------------------------
Hello, thank you for reaching out. I’m sorry to hear your charger hasn’t arrived yet. I’ll look into order ORD12345 right away and will update you as soon as possible. If you have any additional details, please let me know. Thank you for your patience.


In [22]:
word_count = len(reply.split())

print("Word count:", word_count)
print("Maximum allowed:", max_words)

if word_count <= max_words:
    print("Validation passed.")
else:
    print("Reply exceeds the word limit. It needs shortening.")

Word count: 45
Maximum allowed: 60
Validation passed.
